# Estudo Gráfico — notebook de trabalho acadêmico

Este notebook é destinado à construção de **análises e visualizações
definitivas** para um trabalho acadêmico, a partir dos documentos de
declarações e planos de IA já reunidos neste projeto (PBIA, BRICS, OCDE,
AI+, China New Gen, Apply AI Strategy, AI Continent Action Plan, America's
AI Action Plan, entre outros).

Diferente do [`Comparações.ipynb`](./Compara%C3%A7%C3%B5es.ipynb) — que foi
um notebook **exploratório**, usado para gerar ideias e testar rapidamente
diferentes tipos de gráfico — todo código produzido aqui precisa ser
tratado como **material citável**: rigor metodológico, fidelidade ao dado e
ausência de lacunas analíticas não são opcionais.

> **Antes de considerar qualquer análise deste notebook "pronta"**, ela
> precisa passar pelo checklist das células abaixo. Se uma etapa não foi
> verificada, a análise não está pronta — está em rascunho.

## Padrão de rigor exigido

- **Toda afirmação numérica no texto precisa vir de uma célula de código
  executada nesta sessão**, nunca de números aproximados de memória ou
  copiados de fora. Nenhuma síntese em texto deve ser escrita antes de
  rodar o código e conferir a saída real.
- **Todo parâmetro de decisão** (limiares, mínimos de frequência, tamanho
  de amostra de termos, etc.) **precisa ser explícito no código,
  justificado no texto** e, sempre que possível, acompanhado de um teste
  de sensibilidade mostrando que a conclusão não depende de uma escolha
  arbitrária daquele número.
- **Nenhum gráfico entra na versão final sem metodologia clara** explicando
  exatamente o que está sendo medido, de onde vem o dado e que
  transformação foi aplicada (frequência relativa? normalizada por quê?
  contagem bruta?).
- **Toda comparação entre documentos de tamanhos diferentes precisa ser
  normalizada** (frequência relativa, nunca contagem bruta) — e o notebook
  deve declarar isso a cada análise, não deixar implícito.
- **Nenhuma conclusão forte sem checagem de robustez.** Se um resultado só
  se sustenta com um parâmetro específico, isso é uma limitação a ser
  relatada, não uma conclusão a ser afirmada.

## Erros e lacunas metodológicas do `Comparações.ipynb` que NÃO podem se repetir aqui

Este notebook nasceu de uma revisão crítica do `Comparações.ipynb`. Os
problemas abaixo foram identificados lá e **precisam ser endereçados desde
o início** em qualquer análise nova feita aqui — não como correção
posterior:

1. **Tokenização sem lematização/normalização morfológica.** `technology`,
   `technological` e `technologies` foram tratados como termos totalmente
   distintos, fragmentando o sinal de um mesmo conceito e podendo empurrar
   variantes da mesma palavra para classificações diferentes.
   → Usar lematização (ou, no mínimo, agrupamento manual de radicais
   relevantes) antes de qualquer contagem de frequência.

2. **Apenas unigramas — nenhuma expressão composta.** O termo *sovereignty*
   apareceu isolado quando o conceito real no documento era uma expressão
   ("data sovereignty", "technological sovereignty" — título do Eixo 3 do
   PBIA).
   → Extrair também bigramas/trigramas ou colocações relevantes, não só
   palavras soltas.

3. **Heurísticas de limiar em vez de teste estatístico.** Classificações do
   tipo "termo pertence ao bloco X" ou "termo é convergente" usaram razões
   fixas escolhidas por bom senso (ex.: razão 2º/1º colocado ≥ 0,75), sem
   nenhum teste de significância.
   → Preferir estatísticas de *keyness* usadas em linguística de corpus
   (log-likelihood/G², qui-quadrado), que ponderam pela frequência esperada
   dado o tamanho de cada corpus, e reportar o valor estatístico — não só
   a razão bruta.

4. **Gráficos multipainel com escalas de eixo diferentes entre painéis.**
   Isso convida o leitor a comparar visualmente o comprimento de barras que
   na verdade estão em escalas diferentes.
   → Usar sempre a mesma escala (ou valores normalizados/percentuais)
   quando o gráfico for comparar categorias lado a lado.

5. **Rankings que misturam "importância no documento" com "força de
   atração/exclusividade"** sem declarar isso — um termo pode liderar um
   ranking só por ser muito frequente no documento-base, mesmo com atração
   fraca para o grupo em que foi classificado.
   → Sempre que houver uma classificação por "qual lado o termo puxa",
   reportar também a margem/razão que gerou essa classificação, não só a
   contagem.

6. **Ausência de piso mínimo de frequência de referência.** Termos raros
   (poucas ocorrências no documento-base e quase nenhuma nos grupos de
   referência) podem "vencer" uma classificação por *default* quando os
   outros lados são zero — o que pode ser sinal real ou apenas esparsidade
   dos dados.
   → Definir e justificar um piso mínimo de evidência antes de classificar
   qualquer termo, e sinalizar quando um resultado está perto desse piso.

7. **Confusão entre gênero textual/formato do documento e alinhamento
   substantivo.** Palavras como "actions" e "plan" podem puxar um documento
   para um lado só por causa do formato (ex.: um "Action Plan" com lista de
   ações), não por proximidade real de conteúdo.
   → Discutir explicitamente essa possibilidade ao interpretar qualquer
   termo cuja força pareça vir mais do gênero do documento do que do seu
   conteúdo.

8. **Nenhum teste de robustez dos parâmetros escolhidos.** Nenhuma análise
   do `Comparações.ipynb` testou se a conclusão mudava ao variar os
   parâmetros (mínimo de contagem, limiar de convergência, tamanho do
   top-N).
   → Sempre que uma conclusão depender de um parâmetro escolhido à mão,
   testar pelo menos 2–3 valores alternativos e reportar se a conclusão se
   mantém.

---

Sempre que uma nova análise for adicionada a este notebook, revise esta
lista **antes** de escrever a síntese em texto. Se algum dos oito pontos
acima não foi endereçado, marque explicitamente como limitação conhecida
no texto — nunca omita silenciosamente.

---

# 1. Comparação PBIA × 7 documentos internacionais — metodologia e extração de texto

Este notebook compara o **PBIA** (`PBIA.json`) com os sete documentos de
referência já reunidos no projeto: **BRICS** (`BRICS.json`), **OCDE**
(`OCDE.json`), **AI+** (`AI_PLUS.json`), **China New Gen**
(`China_New_Gen.json`), **Apply AI Strategy** (`Apply_AI_Strategy.json`),
**AI Continent Action Plan** (`AI_CONTINENT_ACTION_PLAN.json`) e
**America's AI Action Plan** (`AMERICA_AI_ACTION_PLAN.json`).

O objetivo: medir a similaridade de linguagem/vocabulário entre o PBIA e
cada documento, identificar se compartilham a mesma preferência temática
(direitos humanos? desenvolvimento nacional? segurança? infraestrutura?) e
posicionar o PBIA relativamente aos sete — em termos **relativos**, nunca
absolutos, já que os documentos têm tamanhos muito diferentes entre si.

## 1.1. Por que refazer a extração de texto

O `Comparações.ipynb` extraía texto usando, para a maioria dos documentos,
apenas o campo `sections[].text` de cada JSON. Isso deixava de fora
conteúdo substantivo que está em outros campos — por exemplo, o **PBIA**
tem um campo `responsible_ai_principles` (os princípios de IA responsável
do plano) e um campo `structuring_axes` (os eixos estruturantes, com muito
texto descritivo) que nunca entravam na análise. Isso é particularmente
grave para responder à pergunta sobre foco em direitos humanos — se o
campo que mais provavelmente contém esse vocabulário nunca é lido, a
conclusão fica enviesada antes mesmo de qualquer cálculo.

A tabela abaixo documenta, para cada um dos 8 documentos, exatamente quais
campos do JSON entram no corpus de texto e quais ficam de fora — e por quê.
Cada função de extração abaixo tem comentários explícitos com essa mesma
justificativa, para que o código e a documentação nunca fiquem
dessincronizados.

| Documento | Incluído | Excluído (e por quê) |
|---|---|---|
| **PBIA** | título, objetivos, princípios de IA responsável, pilares, premissas, janelas de oportunidade, linha do tempo de IA, nota de investimentos, setores/ações de impacto imediato, eixos/ações estruturantes, descrição dos exemplos de governança, seções | metadados (data, ISBN, editora, páginas), agregados derivados (`chart_ready_data`, `categories_summary`), nomes próprios de países/instituições nos exemplos de governança, glossário de siglas (nomes de instituições), `references[]` (bibliografia de **outras** obras — não é a voz do PBIA) |
| **BRICS / OCDE** | título + `sections[]` (categoria, subtítulo, texto) | metadados, `categories_summary` (agregado) |
| **AI+** | `sections[]`, campos-chave, metas de desenvolvimento, iniciativas-chave, capacidades básicas de suporte | metadados/nomes próprios de tradutores e editores, `categories_summary` |
| **China New Gen** | título, título completo, preâmbulo, árvore `sections→subsections→items→boxes` (título+parágrafos de cada nível) | nota de tradução (créditos dos **tradutores**, não é conteúdo do documento), estatísticas/linha do tempo agregadas |
| **Apply AI Strategy** | `sections[]`, título completo, componentes do mecanismo de governança, setores dos *flagships* | metadados, `related_initiatives` (nomes próprios de programas), `key_statistics` |
| **AI Continent Action Plan** | `sections[]`, título completo, cinco domínios-chave, títulos das seções principais, estudos de caso, ações-chave | **`extraction_note`** — nota sobre o próprio processo de extração do PDF, não é conteúdo do documento original —, diretório de instalações (`ai_factories`, nomes próprios/locais), `related_initiatives`, `key_statistics` |
| **America's AI Action Plan** | título, subtítulo, citação de abertura, introdução (resumo, pontos-chave, princípios), título da ordem executiva referenciada, pilares/seções/ações recomendadas | nomes e cargos dos autores, `analysis_helpers` (agregado), sigla do órgão responsável por cada ação |

In [ ]:
import json
import re
import math
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

BASE = Path(".")

STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "they", "them", "there",
    "etc",  # abreviação latina "et cetera" — sobrevive à tokenização (é alfabética,
             # tem 3 letras) mas não é vocabulário temático; identificada como ruído
             # ao aparecer entre os termos mais distintivos do cluster China (Seção 7.2)
}
TOKEN_RE = re.compile(r"[a-zA-Z]+(?:-[a-zA-Z]+)*")


def join_parts(parts):
    """Junta uma lista de strings ignorando None (alguns campos de origem são
    opcionais — ex.: AI_PLUS.development_targets[].comparison é None para um
    dos anos, o que quebraria um " ".join ingênuo)."""
    return " ".join(p for p in parts if p)


def all_strings(obj):
    """Recursivamente extrai todo valor string de um objeto dict/list aninhado."""
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, dict):
        for v in obj.values():
            yield from all_strings(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from all_strings(item)

In [ ]:
def extract_pbia(data):
    parts = [data["document"]["title"]]
    parts += data["pbia_objectives"]
    parts += list(all_strings(data["responsible_ai_principles"]))
    parts += list(all_strings(data["pillars"]))
    parts += list(all_strings(data["premises"]))
    parts += list(all_strings(data["windows_of_opportunity"]))
    parts += [t["era"] for t in data["ai_timeline"]]
    parts += [t["event"] for t in data["ai_timeline"]]
    parts.append(data["investments"]["note"])
    parts += data["immediate_impact_sectors"]
    parts += list(all_strings(data["immediate_impact_actions"]))
    parts += list(all_strings(data["structuring_axes"]))
    parts += list(all_strings(data["structuring_actions"]))
    parts += [g["description"] for g in data["governance_examples"]]
    for s in data["sections"]:
        parts.append(s["category"])
        if s.get("subtitle"):
            parts.append(s["subtitle"])
        parts.append(s["text"])
    return join_parts(parts)


def extract_flat_sections(data):
    """Documentos cujo texto está numa lista simples sections[] com campo text."""
    parts = [data["document"]["title"]]
    for s in data["sections"]:
        parts.append(s["category"])
        if s.get("subtitle"):
            parts.append(s["subtitle"])
        parts.append(s["text"])
    return parts


def extract_brics(data):
    return join_parts(extract_flat_sections(data))


def extract_ocde(data):
    return join_parts(extract_flat_sections(data))


def extract_aiplus(data):
    parts = extract_flat_sections(data)
    parts += data["key_fields"]
    for t in data["development_targets"]:
        parts.append(t["comparison"])
        parts += t["milestones"]
    for k in data["key_initiatives"]:
        parts.append(k["title"])
        parts += k["items"]
    parts += [b["title"] for b in data["basic_support_capabilities"]]
    return join_parts(parts)


def extract_china_new_gen(data):
    """Texto em árvore (sections -> subsections -> items -> boxes), cada nível
    com title/paragraphs — requer travessia recursiva própria."""
    parts = [data["document"]["title"], data["document"]["full_title"], data["document"].get("preamble", "")]

    def walk(node, parts):
        if "title" in node:
            parts.append(node["title"])
        if "paragraphs" in node:
            parts.extend(node["paragraphs"])
        for key in ("subsections", "items", "boxes"):
            for child in node.get(key, []):
                walk(child, parts)

    for chapter in data["sections"]:
        walk(chapter, parts)
    return join_parts(parts)


def extract_apply_ai_strategy(data):
    parts = extract_flat_sections(data)
    parts.append(data["document"]["full_title"])
    for c in data["governance_mechanism"]["components"]:
        parts.append(c["name"])
        parts.append(c["role"])
    parts += [s["sector"] for s in data["sectoral_flagships_summary"]]
    return join_parts(parts)


def extract_ai_continent(data):
    parts = extract_flat_sections(data)
    parts.append(data["document"]["full_title"])
    parts += data["document"]["five_key_domains"]
    parts += data["document"]["main_sections"]
    for cs in data["case_studies"]:
        parts.append(cs["title"])
        parts.append(cs["text"])
    for a in data["key_actions"]:
        parts.append(a["action"])
        parts.append(a["section"])
    return join_parts(parts)


def extract_america(data):
    meta = data["document_metadata"]
    parts = [meta["title"], meta["subtitle"], meta["opening_quote"]["text"]]
    intro = data["introduction"]
    parts.append(intro["summary"])
    parts += intro["key_points"]
    parts += intro["cross_cutting_principles"]
    parts.append(intro["referenced_executive_order"]["title"])
    for pillar in data["pillars"]:
        parts.append(pillar["title"])
        parts.append(pillar["summary"])
        for section in pillar["sections"]:
            parts.append(section["section_title"])
            parts.append(section["summary"])
            for action in section["recommended_policy_actions"]:
                parts.append(action["action"])
    return join_parts(parts)


EXTRACTORS = {
    "PBIA.json": extract_pbia,
    "BRICS.json": extract_brics,
    "OCDE.json": extract_ocde,
    "AI_PLUS.json": extract_aiplus,
    "China_New_Gen.json": extract_china_new_gen,
    "Apply_AI_Strategy.json": extract_apply_ai_strategy,
    "AI_CONTINENT_ACTION_PLAN.json": extract_ai_continent,
    "AMERICA_AI_ACTION_PLAN.json": extract_america,
}

LABELS = {
    "PBIA.json": "PBIA (Brasil)",
    "BRICS.json": "BRICS Declaration",
    "OCDE.json": "OECD Recommendation",
    "AI_PLUS.json": "AI+ (China, 2023)",
    "China_New_Gen.json": "China New Gen (2017)",
    "Apply_AI_Strategy.json": "Apply AI Strategy (UE)",
    "AI_CONTINENT_ACTION_PLAN.json": "AI Continent Action Plan (UE)",
    "AMERICA_AI_ACTION_PLAN.json": "America's AI Action Plan (EUA)",
}

# Paleta categórica validada (skill dataviz — scripts/validate_palette.js), ordem
# fixa, uma cor por documento, nunca reciclada entre gráficos. Todo gráfico deste
# notebook também rotula cada documento diretamente (eixo Y ou anotação), então a
# identidade nunca depende só da cor.
COLOR = {
    "PBIA.json": "#e34948",                      # red — cor de identidade do PBIA
    "BRICS.json": "#2a78d6",                      # blue
    "AMERICA_AI_ACTION_PLAN.json": "#008300",     # green
    "China_New_Gen.json": "#e87ba4",              # magenta
    "AI_CONTINENT_ACTION_PLAN.json": "#eda100",   # yellow
    "OCDE.json": "#1baf7a",                       # aqua
    "Apply_AI_Strategy.json": "#eb6834",          # orange
    "AI_PLUS.json": "#4a3aa7",                    # violet
}

SURFACE, INK_PRIMARY, INK_MUTED, GRIDLINE = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

raw_text = {}
for fn, extractor in EXTRACTORS.items():
    data = json.load(open(BASE / fn, encoding="utf-8"))
    raw_text[fn] = extractor(data)

doc_order = list(EXTRACTORS.keys())  # PBIA primeiro, depois os 7 de referência
ref_order = doc_order[1:]

for fn in doc_order:
    print(f"{LABELS[fn]:32s} {len(raw_text[fn]):7d} caracteres  ~{len(raw_text[fn].split()):5d} palavras (bruto)")

### Gráfico — volume de texto extraído por documento

Antes de qualquer normalização, é importante visualizar a disparidade de
tamanho entre os documentos — é exatamente essa disparidade que torna
obrigatório usar frequência **relativa** (por 1.000 tokens) em toda
comparação daqui em diante, nunca contagem bruta.

In [ ]:
word_counts = {fn: len(raw_text[fn].split()) for fn in doc_order}
fig, ax = plt.subplots(figsize=(8, 5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
y_pos = range(len(doc_order))
labels_sorted = sorted(doc_order, key=lambda fn: word_counts[fn])
values_sorted = [word_counts[fn] for fn in labels_sorted]
colors_sorted = [COLOR[fn] for fn in labels_sorted]
ax.barh(list(y_pos), values_sorted, color=colors_sorted, height=0.6, zorder=3)
for y, v in zip(y_pos, values_sorted):
    ax.text(v + max(values_sorted) * 0.015, y, f"{v:,}".replace(",", "."), va="center", ha="left",
            color=INK_PRIMARY, fontsize=9)
ax.set_yticks(list(y_pos))
ax.set_yticklabels([LABELS[fn] for fn in labels_sorted], color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Palavras extraídas (bruto, antes de tokenizar)", color=INK_MUTED, fontsize=10)
ax.set_title("Volume de texto por documento — por isso toda comparação usa frequência relativa",
             color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=14)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

ratio = word_counts["PBIA.json"] / word_counts["OCDE.json"]
print(f"Maior documento (PBIA) tem {ratio:.1f}x mais palavras que o menor (OCDE) — confirma que comparar contagem bruta seria inválido.")

---

# 2. Tokenização e normalização morfológica

## 2.1. Tokenização

Mesma tokenização usada no `Comparações.ipynb` — regex `[a-zA-Z]+(?:-[a-zA-Z]+)*`
em minúsculas, descartando *stopwords* e palavras com ≤2 letras. Essa parte
funcionou bem e não muda aqui.

## 2.2. Stemmer próprio, documentado (por que existe)

O `Comparações.ipynb` tratava `technology`, `technological` e `technologies`
como três termos totalmente distintos, fragmentando o sinal de um mesmo
conceito. Para corrigir isso sem depender de uma biblioteca externa (NLTK
não está instalada, e o download de dados do WordNet dependeria de acesso a
um servidor externo não testado), este notebook usa uma abordagem **híbrida
e deliberadamente conservadora**:

1. Um **dicionário curado** (`MANUAL_STEM_OVERRIDES`) para as famílias
   morfológicas de maior impacto no corpus — identificadas empiricamente ao
   agrupar o vocabulário combinado dos 8 documentos por prefixo compartilhado
   e inspecionar as famílias com mais de ~100 ocorrências combinadas (ex.:
   *development/develop/developing* somam 610 ocorrências). Cada entrada é
   100% auditável — é só ler o dicionário abaixo.
2. Um conjunto pequeno de **regras genéricas de sufixo** (sempre remoção
   pura, nunca substituição irregular) para o restante do vocabulário —
   cobre plurais e os sufixos derivacionais mais comuns do inglês
   (*-tion, -ment, -ity, -al, -ive*, etc.), mas **deliberadamente não mexe
   em -ing/-ed** (verbos no gerúndio/passado): a restauração do "e" mudo e
   de consoantes dobradas (ex.: *using→use*, *running→run*) é a parte mais
   propensa a erro de qualquer stemmer de sufixos simples, e preferimos não
   normalizar a normalizar errado silenciosamente.

**Limitação conhecida e documentada**: por não tratar -ing/-ed de forma
genérica, verbos nessas formas só são unificados com sua raiz quando estão
no dicionário curado. O vocabulário fora dele mantém essas variantes
separadas — um viés conservador, não um erro silencioso.

**Validação**: antes de aplicar este stemmer a qualquer análise, ele foi
testado contra 28 famílias morfológicas do vocabulário real do corpus
(incluindo as usadas no léxico temático da Seção 6, onde a precisão importa
mais) — a validação e as correções feitas estão documentadas nos comentários
do dicionário abaixo.

In [ ]:
MANUAL_STEM_OVERRIDES = {
    # cada família abaixo foi identificada por ocorrência combinada >100 no
    # vocabulário conjunto dos 8 documentos (ver auditoria de prefixos)
    "development": "develop", "developing": "develop", "developed": "develop",
    "developments": "develop", "developers": "develop", "developer": "develop",
    "development-oriented": "develop",

    "technology": "technolog", "technologies": "technolog", "technological": "technolog",
    "technologically": "technolog", "technology-led": "technolog",

    "intelligence": "intelligen", "intelligent": "intelligen",
    "intelligentized": "intelligen", "intelligentization": "intelligen",
    "intelligentization-based": "intelligen", "intelligence-based": "intelligen",
    "intelligence-driven": "intelligen",

    "innovation": "innovat", "innovative": "innovat", "innovations": "innovat",
    "innovate": "innovat", "innovators": "innovat", "innovating": "innovat",
    "innovation-oriented": "innovat", "innovation-driven": "innovat",
    "innovation-style": "innovat", "innovation-friendly": "innovat",

    "industry": "industr", "industrial": "industr", "industries": "industr",
    "industry-specific": "industr", "industry-driven": "industr",
    "industry-education": "industr", "industry-leading": "industr",

    "governance": "govern", "government": "govern", "governmental": "govern",
    "governments": "govern", "governing": "govern", "government-procured": "govern",

    "security": "secur", "secure": "secur", "securely": "secur",
    "securing": "secur", "secured": "secur", "secure-by-design": "secur",
    "security-related": "secur",

    "sustainable": "sustain", "sustainability": "sustain", "sustained": "sustain",

    "responsible": "respons", "responsibility": "respons",
    "responsibilities": "respons", "responsibly": "respons",

    "strategic": "strateg", "strategy": "strateg", "strategies": "strateg",

    "transparency": "transpar", "transparent": "transpar", "transparently": "transpar",

    "cooperation": "cooperat", "cooperative": "cooperat", "cooperate": "cooperat",

    "investment": "invest", "investing": "invest", "investments": "invest",

    "creation": "creat", "creative": "creat", "creating": "creat",
    "creates": "creat", "created": "creat", "creativity": "creat",
    "creators": "creat", "create": "creat", "creational": "creat",

    "using": "use", "used": "use", "uses": "use",

    "ensuring": "ensur", "ensure": "ensur", "ensures": "ensur", "ensured": "ensur",

    "increasing": "increas", "increase": "increas", "increased": "increas",
    "increasingly": "increas", "increases": "increas", "incrementally": "increas",

    "promoting": "promot", "promote": "promot", "promotion": "promot",
    "promotes": "promot", "promoted": "promot", "promotable": "promot",

    "improving": "improv", "improve": "improv", "improvement": "improv",
    "improved": "improv", "improves": "improv",

    "strengthening": "strength", "strengthen": "strength",
    "strengthened": "strength", "strengthens": "strength",

    "generation": "generat", "generative": "generat", "generate": "generat",
    "generating": "generat", "generated": "generat", "generates": "generat",
    "generalization": "generat",

    "integration": "integrat", "integrated": "integrat",
    "integrating": "integrat", "integrate": "integrat", "integrative": "integrat",

    "regulation": "regulat", "regulatory": "regulat", "regulations": "regulat",
    "regulate": "regulat", "regulated": "regulat", "regulating": "regulat",

    "production": "produc", "products": "produc", "productivity": "produc",
    "product": "produc", "productive": "produc", "producers": "produc",
    "produced": "produc", "produce": "produc", "producer": "produc",
    "producing": "produc", "productions": "produc",

    "computing": "comput", "computational": "comput", "compute": "comput",
    "computer": "comput", "computerization": "comput", "computers": "comput",
    "computable": "comput",

    "advanced": "advanc", "advance": "advanc", "advances": "advanc",
    "advancing": "advanc", "advancement": "advanc", "advancements": "advanc",

    "digital": "digit", "digitalisation": "digit", "digitalized": "digit",
    "digital-intense": "digit", "digitally": "digit", "digitization": "digit",

    # Famílias adicionadas para dar suporte ao léxico temático da Seção 5 (rigor
    # exigido lá é maior — cada forma de superfície foi conferida manualmente contra
    # o vocabulário real do corpus, não apenas a lista genérica de sufixos abaixo).
    "discrimination": "discriminat", "discriminatory": "discriminat",
    "discriminate": "discriminat", "discriminated": "discriminat", "discriminates": "discriminat",
    "protect": "protect", "protects": "protect", "protected": "protect",
    "protecting": "protect", "protection": "protect", "protections": "protect",
    "protective": "protect",
    "bias": "bias", "biased": "bias", "biases": "bias",
    "diverse": "divers", "diversity": "divers", "diversify": "divers", "diversified": "divers",
    "equity": "equit", "equitable": "equit", "equitably": "equit",
    "accountability": "account", "accountable": "account", "accountably": "account",
    "inclusion": "inclus", "inclusive": "inclus", "inclusivity": "inclus", "inclusiveness": "inclus",
    "trustworthy": "trust", "trusted": "trust", "trusting": "trust", "trustworthiness": "trust",
    "fairness": "fair", "fairly": "fair",
    "safety": "safe", "safely": "safe", "unsafe": "unsafe",
    "vulnerable": "vulner", "vulnerability": "vulner", "vulnerabilities": "vulner",
    "consented": "consent", "consenting": "consent",
    "growth": "grow", "growing": "grow", "grows": "grow", "grew": "grow",
    "competitiveness": "competit", "competitive": "competit", "competition": "competit",
    "compete": "competit", "competing": "competit",
    "employment": "employ", "employed": "employ", "employer": "employ", "employers": "employ",
    "trading": "trade", "trader": "trade", "traders": "trade",
    "sovereignty": "sovereign",
    "defense": "defens", "defence": "defens", "defensive": "defens",
    "defend": "defens", "defended": "defens", "defending": "defens",
    "dominance": "domin", "dominant": "domin", "dominate": "domin", "dominated": "domin",
    "geopolitical": "geopolit", "geopolitics": "geopolit",
    "oversight": "oversee", "overseeing": "oversee",
    "institutional": "institution", "institutions": "institution",
    "compliance": "compl", "compliant": "compl", "comply": "compl", "complies": "compl",
    "semiconductors": "semiconductor",
    "supercomputers": "supercomputer", "supercomputing": "supercomputer",
    "skills": "skill", "skilled": "skill",
}

# Sufixos genéricos: (sufixo, mínimo de letras que precisam sobrar após remover).
# Aplicados em ordem — o primeiro que casar "vence" — e SEMPRE por remoção pura
# (nunca substituição irregular), o que evita erros de "e" mudo/consoante dobrada.
GENERIC_SUFFIXES = [
    ("ies", 3, "y"),       # technologies -> technology (substituição regular e seg.)
    ("ications", 4, ""), ("ication", 4, ""),
    ("izations", 4, ""), ("ization", 4, ""),
    ("ities", 4, ""), ("ity", 4, ""),
    ("ances", 4, ""), ("ance", 4, ""),
    ("ences", 4, ""), ("ence", 4, ""),
    ("ments", 4, ""), ("ment", 4, ""),
    ("nesses", 4, ""), ("ness", 4, ""),
    ("atively", 4, ""), ("ative", 4, ""),
    ("ically", 4, ""), ("ical", 4, ""),
    ("ives", 4, ""), ("ive", 4, ""),
    ("ally", 4, ""),
    ("ously", 4, ""), ("ous", 4, ""),
    ("ful", 4, ""),
    ("ably", 4, ""), ("ibly", 4, ""), ("able", 4, ""), ("ible", 4, ""),
    ("ancy", 4, ""), ("ency", 4, ""),
    ("als", 4, ""), ("al", 4, ""),
]

# Palavras que NÃO devem sofrer a remoção simples de "s" final de plural
# (terminam em -ss/-us/-is/-ous e a remoção ingênua as corromperia:
# "process"->"proces", "status"->"statu", "analysis"->"analysi", "various"->"variou")
_PLURAL_S_EXCEPTIONS_SUFFIXES = ("ss", "us", "is", "ous")


def stem(word):
    """Normaliza uma palavra já tokenizada (minúscula, alfabética) à sua forma
    aproximada de raiz. Ver documentação do módulo para a lógica completa."""
    if word in MANUAL_STEM_OVERRIDES:
        return MANUAL_STEM_OVERRIDES[word]

    for suffix, min_len, replacement in GENERIC_SUFFIXES:
        if word.endswith(suffix) and len(word) - len(suffix) >= min_len:
            return word[: -len(suffix)] + replacement

    if word.endswith("s") and not word.endswith(_PLURAL_S_EXCEPTIONS_SUFFIXES) and len(word) > 3:
        return word[:-1]

    return word


### Validação empírica do stemmer

Antes/depois para as famílias morfológicas de maior impacto no corpus —
confirma que o dicionário curado está unificando corretamente, sem juntar
palavras de raízes diferentes.

In [ ]:
def tokenize(text, use_stem=True):
    toks = TOKEN_RE.findall(text.lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 2]
    if use_stem:
        toks = [stem(t) for t in toks]
    return toks


def load_all(use_stem=True):
    tokens, counts = {}, {}
    for fn in doc_order:
        toks = tokenize(raw_text[fn], use_stem=use_stem)
        tokens[fn] = toks
        counts[fn] = Counter(toks)
    return tokens, counts


tokens_new, counts_new = load_all(use_stem=True)

# validação: agrupa o vocabulário combinado (sem stem) por família e mostra a
# unificação obtida para as famílias de maior impacto
all_raw_tokens = Counter()
for fn in doc_order:
    all_raw_tokens.update(tokenize(raw_text[fn], use_stem=False))

for fam in ["develop", "technolog", "intelligen", "innovat", "govern", "secur", "industr"]:
    variants = sorted([(w, c) for w, c in all_raw_tokens.items() if stem(w) == fam], key=lambda x: -x[1])
    total = sum(c for _, c in variants)
    print(f"{fam:12s} ({total:4d} ocorrências) <- {variants}")

print()
for fn in doc_order:
    print(f"{LABELS[fn]:32s} {len(tokens_new[fn]):6d} tokens (pós-stem)  {len(counts_new[fn]):5d} termos únicos")

---

# 3. Similaridade de linguagem — PBIA × cada um dos 7 documentos

## Metodologia (o que já funcionava bem no `Comparações.ipynb` e foi mantido)

- **Frequência relativa** (ocorrências por 1.000 tokens), nunca contagem
  bruta — neutraliza a diferença de tamanho entre os documentos (Seção 1.1).
- **Similaridade de cosseno** sobre os vetores de frequência relativa — mede
  o quanto os dois documentos têm a **mesma ênfase proporcional** no mesmo
  vocabulário (é a métrica mais informativa sobre "preferência temática").
- **Índice de Jaccard** — proporção de termos que aparecem nos dois
  documentos sobre o total de termos distintos de ambos — mede sobreposição
  pura de vocabulário, sem pesar frequência (métrica complementar).

As duas métricas usam agora o corpus completo (Seção 1) e o vocabulário
normalizado por stem (Seção 2), não apenas `sections[].text` sem
normalização.

In [ ]:
def relative_freq(counter, total, term):
    return counter.get(term, 0) / total * 1000


def cosine_sim(counts_a, total_a, counts_b, total_b):
    vocab = set(counts_a) | set(counts_b)
    va = [relative_freq(counts_a, total_a, t) for t in vocab]
    vb = [relative_freq(counts_b, total_b, t) for t in vocab]
    dot = sum(x * y for x, y in zip(va, vb))
    na = math.sqrt(sum(x * x for x in va))
    nb = math.sqrt(sum(y * y for y in vb))
    return dot / (na * nb) if na and nb else 0.0


def jaccard_sim(counts_a, counts_b):
    va, vb = set(counts_a), set(counts_b)
    return len(va & vb) / len(va | vb)


pbia_counts, pbia_total = counts_new["PBIA.json"], len(tokens_new["PBIA.json"])
results_new = {}
for fn in ref_order:
    cos = cosine_sim(pbia_counts, pbia_total, counts_new[fn], len(tokens_new[fn]))
    jac = jaccard_sim(pbia_counts, counts_new[fn])
    results_new[fn] = (cos, jac)
    print(f"PBIA x {LABELS[fn]:32s} cosseno={cos:.4f}  jaccard={jac:.4f}")

### Gráfico — similaridade do PBIA com cada documento

Mesma escala em todo o eixo X dentro de cada painel (cosseno à esquerda,
Jaccard à direita) — ao contrário do gráfico multipainel de blocos do
`Comparações.ipynb`, aqui não há o risco de comparar visualmente barras em
escalas diferentes, porque é um único painel por métrica.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=SURFACE)

for ax, metric_idx, metric_name in [(axes[0], 0, "Similaridade de cosseno (ênfase)"),
                                     (axes[1], 1, "Índice de Jaccard (vocabulário)")]:
    ax.set_facecolor(SURFACE)
    ranked = sorted(ref_order, key=lambda fn: results_new[fn][metric_idx])
    values = [results_new[fn][metric_idx] for fn in ranked]
    colors = [COLOR[fn] for fn in ranked]
    y_pos = range(len(ranked))
    ax.barh(list(y_pos), values, color=colors, height=0.6, zorder=3)
    for y, v in zip(y_pos, values):
        ax.text(v + max(values) * 0.02, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([LABELS[fn] for fn in ranked], color=INK_PRIMARY, fontsize=9.5)
    ax.set_xlim(0, max(values) * 1.28)
    ax.set_xlabel(metric_name, color=INK_MUTED, fontsize=9.5)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle("Com qual documento o PBIA tem mais similaridade de linguagem?",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

---

# 4. Checagem de robustez — os resultados mudam com a metodologia?

Um dos erros do `Comparações.ipynb` era não testar se as conclusões se
mantinham sob escolhas metodológicas diferentes. Aqui reconstruímos a
extração **antiga** (só `sections[].text`, sem stemming — reproduzindo
exatamente o método anterior) e comparamos o ranking de similaridade
resultante com o do pipeline novo (Seções 1–3).

In [ ]:
def extract_old_flat(data):
    return join_parts(s["text"] for s in data["sections"])


def extract_old_china(data):
    parts = [data["document"]["title"], data["document"].get("preamble", "")]

    def walk(node, parts):
        if "title" in node:
            parts.append(node["title"])
        if "paragraphs" in node:
            parts.extend(node["paragraphs"])
        for key in ("subsections", "items", "boxes"):
            for child in node.get(key, []):
                walk(child, parts)

    for chapter in data["sections"]:
        walk(chapter, parts)
    return join_parts(parts)


def extract_old_america(data):
    meta = data["document_metadata"]
    parts = [meta["title"], meta["subtitle"], meta["opening_quote"]["text"]]
    intro = data["introduction"]
    parts.append(intro["summary"])
    parts += intro["key_points"]
    parts += intro["cross_cutting_principles"]
    for pillar in data["pillars"]:
        parts.append(pillar["title"])
        parts.append(pillar["summary"])
        for section in pillar["sections"]:
            parts.append(section["section_title"])
            parts.append(section["summary"])
            for action in section["recommended_policy_actions"]:
                parts.append(action["action"])
    return join_parts(parts)


OLD_EXTRACTORS = {fn: extract_old_flat for fn in doc_order}
OLD_EXTRACTORS["China_New_Gen.json"] = extract_old_china
OLD_EXTRACTORS["AMERICA_AI_ACTION_PLAN.json"] = extract_old_america

tokens_old, counts_old = {}, {}
for fn, extractor in OLD_EXTRACTORS.items():
    data = json.load(open(BASE / fn, encoding="utf-8"))
    toks = tokenize(extractor(data), use_stem=False)
    tokens_old[fn] = toks
    counts_old[fn] = Counter(toks)

results_old = {}
for fn in ref_order:
    cos = cosine_sim(counts_old["PBIA.json"], len(tokens_old["PBIA.json"]), counts_old[fn], len(tokens_old[fn]))
    results_old[fn] = cos

rank_new = sorted(ref_order, key=lambda fn: -results_new[fn][0])
rank_old = sorted(ref_order, key=lambda fn: -results_old[fn])
for i, fn in enumerate(rank_new, 1):
    print(f"{i}. {LABELS[fn]:32s} novo={results_new[fn][0]:.3f}   antigo={results_old[fn]:.3f}   posição antiga={rank_old.index(fn)+1}")

### Gráfico — o ranking de proximidade muda entre metodologias?

Gráfico de inclinação (*slope chart*): cada linha liga a posição de um
documento no ranking antigo (esquerda) à sua posição no ranking novo
(direita). Linha quase horizontal = posição estável; linha inclinada =
posição sensível à metodologia — e por isso deve ser tratada como uma
conclusão menos firme no texto.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for fn in ref_order:
    y_old = rank_old.index(fn) + 1
    y_new = rank_new.index(fn) + 1
    ax.plot([0, 1], [y_old, y_new], color=COLOR[fn], linewidth=2.2, zorder=3, alpha=0.85,
            marker="o", markersize=7, markeredgecolor=SURFACE, markeredgewidth=1)
    ax.text(-0.03, y_old, LABELS[fn], ha="right", va="center", fontsize=9.5, color=INK_PRIMARY)
    ax.text(1.03, y_new, LABELS[fn], ha="left", va="center", fontsize=9.5, color=INK_PRIMARY)

ax.set_xlim(-0.55, 1.55)
ax.set_ylim(7.6, 0.4)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Metodologia ANTIGA\n(só sections[], sem stemming)",
                     "Metodologia NOVA\n(extração completa + stemming)"],
                    color=INK_PRIMARY, fontsize=10, fontweight="bold")
ax.set_yticks(range(1, 8))
ax.set_ylabel("Posição no ranking de similaridade com o PBIA (1 = mais parecido)", color=INK_MUTED, fontsize=9.5)
ax.set_title("O ranking de proximidade com o PBIA muda conforme a metodologia?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(left=False, bottom=False)

fig.tight_layout()
plt.show()

---

# 5. Vocabulário distintivo do PBIA — teste estatístico de *keyness*

O `Comparações.ipynb` usava uma heurística de razão (2º/1º colocado ≥ 0,75)
para dizer se um termo "pertencia" a um grupo — sem nenhum teste de
significância. Aqui usamos **log-likelihood ratio (G²)**, a estatística
padrão de *keyness* em linguística de corpus (Dunning, 1993), que compara a
frequência observada de cada termo no PBIA com a esperada sob a hipótese de
que PBIA e o corpus de referência usam esse termo na mesma proporção —
ponderando pelo tamanho de cada corpus. G² > 10,83 corresponde a p < 0,001.

**Corpus de referência**: os sete documentos combinados em um único corpus
(pool), para que a pergunta seja "o que torna o vocabulário do PBIA
distinto do conjunto de referência internacional como um todo" — e não sete
comparações par a par, o que fragmentaria demais a leitura.

**Piso mínimo de evidência**: só termos com frequência combinada (PBIA +
pool) ≥ 5 entram no teste — evita que termos raríssimos, onde qualquer
diferença é estatisticamente instável, dominem a lista.

**Termos autorreferentes excluídos da leitura** (mas não do cálculo de
similaridade das seções 3–4, onde são um sinal válido): nomes do próprio
país/instituição de cada documento não são vocabulário "temático" — são
triviais (é óbvio que o PBIA menciona "Brasil" mais que os outros).

In [ ]:
SELF_REFERENTIAL_KEYNESS = {
    "pbia", "brazil", "brazilian", "sus", "finep",
    "china", "chinese", "european", "europe", "commission", "eu",
    "america", "american", "united", "states", "trump", "brics", "ocde", "oecd",
}


def log_likelihood(a, c, b, d):
    """a = freq do termo no corpus A; c = total de tokens do corpus A
    b = freq do termo no corpus B; d = total de tokens do corpus B
    Retorna G² (estatística de razão de verossimilhança, ~qui-quadrado, 1 gl)."""
    if a == 0 and b == 0:
        return 0.0
    e1 = c * (a + b) / (c + d)
    e2 = d * (a + b) / (c + d)
    g2 = 0.0
    if a > 0:
        g2 += a * math.log(a / e1)
    if b > 0:
        g2 += b * math.log(b / e2)
    return 2 * g2


pooled_counts = Counter()
pooled_total = 0
for fn in ref_order:
    pooled_counts += counts_new[fn]
    pooled_total += len(tokens_new[fn])

MIN_COUNT_KEYNESS = 5
keyness_rows = []
for t in set(pbia_counts) | set(pooled_counts):
    if t in SELF_REFERENTIAL_KEYNESS:
        continue
    a, b = pbia_counts.get(t, 0), pooled_counts.get(t, 0)
    if a + b < MIN_COUNT_KEYNESS:
        continue
    g2 = log_likelihood(a, pbia_total, b, pooled_total)
    direction = "PBIA" if (a / pbia_total) > (b / pooled_total) else "outros"
    keyness_rows.append((t, a, b, g2, direction))

keyness_rows.sort(key=lambda r: -r[3])
n_sig = sum(1 for r in keyness_rows if r[3] > 10.83)
print(f"{len(keyness_rows)} termos testados (freq. combinada >= {MIN_COUNT_KEYNESS}, autorreferentes excluídos)")
print(f"{n_sig} termos com G² > 10,83 (p<0,001)")
print()
for r in keyness_rows[:15]:
    print(f"  {r[0]:14s} G²={r[3]:7.1f}  PBIA={r[1]:4d}  outros(pool)={r[2]:4d}  puxa p/ {r[4]}")

### Gráfico — termos mais distintivos do PBIA (G², estatisticamente significativos)

Os termos mais super-representados no PBIA (barras à direita) e os mais
sub-representados — ou seja, mais usados pelo conjunto dos outros sete
documentos (barras à esquerda) — entre os estatisticamente significativos
(G² > 10,83, p < 0,001). O comprimento da barra é o próprio G² (a força da
evidência estatística), e a frequência real de cada termo em cada lado é
anotada ao lado da barra — assim a força estatística e a frequência bruta
aparecem juntas, nunca uma no lugar da outra.

In [ ]:
TOP_N_KEYNESS = 12
pbia_side = [r for r in keyness_rows if r[4] == "PBIA"][:TOP_N_KEYNESS]
outros_side = [r for r in keyness_rows if r[4] == "outros"][:TOP_N_KEYNESS]

fig, ax = plt.subplots(figsize=(10, 7), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

rows_plot = outros_side[::-1] + pbia_side[::-1]
y_pos = range(len(rows_plot))
signed_g2 = [-r[3] if r[4] == "outros" else r[3] for r in rows_plot]
colors = [COLOR["AI_CONTINENT_ACTION_PLAN.json"] if r[4] == "outros" else COLOR["PBIA.json"] for r in rows_plot]

ax.barh(list(y_pos), signed_g2, color=colors, height=0.62, zorder=3)
# rótulo de frequência SEMPRE dentro da própria barra (nunca além da ponta) — evita
# colidir com os rótulos do eixo Y quando a barra é curta (ver Seção 5, revisão)
for y, r, g in zip(y_pos, rows_plot, signed_g2):
    freq_txt = f"{r[1]}× | {r[2]}×"
    xpos = g * 0.97
    ha = "right" if g >= 0 else "left"
    ax.text(xpos, y, freq_txt, va="center", ha=ha, fontsize=8, color="white", fontweight="bold")

ax.set_yticks(list(y_pos))
ax.set_yticklabels([r[0] for r in rows_plot], color=INK_PRIMARY, fontsize=9.5)
ax.axvline(0, color=GRIDLINE, linewidth=1.2, zorder=2)
ax.set_xlabel("G² (log-likelihood) — negativo: puxa para os outros 7 · positivo: puxa para o PBIA\n(rótulos dentro das barras: ocorrências PBIA × | pool-dos-outros ×)",
              color=INK_MUTED, fontsize=9)
ax.set_title("Vocabulário mais distintivo do PBIA vs. pool dos outros 7 documentos (G² significativo, p<0,001)",
             color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14)
pad = max(abs(v) for v in signed_g2) * 0.15
ax.set_xlim(min(signed_g2) - pad, max(signed_g2) + pad)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

In [ ]:
# Mesmos dados do gráfico técnico acima (keyness_rows), numa versão pensada para
# leitura rápida: sem G²/p-valor, com o termo em português + o original em inglês,
# e com o eixo em algo concreto ("vezes mais comum") em vez de uma estatística.
# Curadoria manual de 8 termos por lado — os mais claros e menos redundantes dos
# 116 termos estatisticamente significativos já listados acima.

DISPLAY_TERMS = [
    # (stem, rótulo em português, palavra original em destaque, lado)
    ("program", "Programas", "program", "PBIA"),
    ("reduction", "Redução", "reduction", "PBIA"),
    ("health", "Saúde", "health", "PBIA"),
    ("qualif", "Qualificação profissional", "qualification", "PBIA"),
    ("increas", "Aumento", "increase", "PBIA"),
    ("artifici", "Artificial", "artificial", "PBIA"),
    ("evasion", "Evasão escolar", "evasion", "PBIA"),
    ("population", "População", "population", "PBIA"),
    ("smart", "\u201cSmart\u201d (inteligente)", "smart", "outros"),
    ("factory", "Fábricas", "factory", "outros"),
    ("accelerate", "Acelerar", "accelerate", "outros"),
    ("build", "Construir", "build", "outros"),
    ("breakthrough", "Avanços revolucionários", "breakthrough", "outros"),
    ("deploy", "Implantação", "deployment", "outros"),
    ("manufacturing", "Manufatura", "manufacturing", "outros"),
    ("safe", "Segurança (safety)", "safety", "outros"),
]

keyness_by_stem = {r[0]: r for r in keyness_rows}

rows = []
for stem_key, label_pt, label_en, side in DISPLAY_TERMS:
    t, a, b, g2, direction = keyness_by_stem[stem_key]
    rate_pbia = a / pbia_total * 1000
    rate_pool = b / pooled_total * 1000
    rows.append((label_pt, label_en, side, rate_pbia, rate_pool, a, b))

fig, ax = plt.subplots(figsize=(11, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

OUTROS_COLOR = "#6b7280"  # cinza neutro — representa o conjunto dos 7 documentos, não um documento específico

rows_plot = [r for r in rows if r[2] == "outros"][::-1] + [r for r in rows if r[2] == "PBIA"][::-1]
y_pos = range(len(rows_plot))
values = [-r[4] if r[2] == "outros" else r[3] for r in rows_plot]
colors = [OUTROS_COLOR if r[2] == "outros" else COLOR["PBIA.json"] for r in rows_plot]

ax.barh(list(y_pos), values, color=colors, height=0.66, zorder=3)

ylabels = []
for r in rows_plot:
    ylabels.append(f"{r[0]}\n({r[1]})")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(ylabels, color=INK_PRIMARY, fontsize=11)

for y, r, v in zip(y_pos, rows_plot, values):
    label_pt, label_en, side, rate_pbia, rate_pool, a, b = r
    if side == "PBIA":
        if b == 0:
            txt = "só aparece no PBIA"
        else:
            txt = f"{rate_pbia / rate_pool:.0f}x mais comum no PBIA"
    else:
        if a == 0:
            txt = "o PBIA nunca usa esta palavra"
        else:
            txt = f"{rate_pool / rate_pbia:.0f}x mais comum nos outros 7"
    ha = "left" if v >= 0 else "right"
    xpos = v + (max(values) * 0.02 if v >= 0 else min(values) * 0.02)
    ax.text(xpos, y, txt, va="center", ha=ha, fontsize=10.5, color=INK_PRIMARY, fontweight="bold")

ax.axvline(0, color=GRIDLINE, linewidth=1.4, zorder=2)
pad = max(abs(v) for v in values) * 1.55
ax.set_xlim(-pad, pad)
ax.set_xticks([])
ax.set_xlabel("← palavras dos outros 7 documentos          palavras do PBIA →",
              color=INK_MUTED, fontsize=11, fontweight="bold")
ax.set_title("As palavras que mais diferenciam o PBIA dos outros sete documentos",
             color=INK_PRIMARY, fontsize=15, fontweight="bold", pad=18)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False)

fig.tight_layout()
plt.show()

### O que este gráfico mostra

Este gráfico usa exatamente os mesmos dados e o mesmo teste estatístico do
gráfico técnico acima — só muda a forma de apresentar. Em vez da
estatística G² (que exige saber o que é um teste de significância), cada
barra mostra **quantas vezes mais comum** aquela palavra é de um lado do
que do outro — um número mais fácil de sentir no corpo.

**Do lado direito (vermelho)**: palavras que o PBIA usa muito mais do que
os outros sete documentos juntos. **Do lado esquerdo (cinza)**: palavras
que os outros sete documentos usam muito mais do que o PBIA — em alguns
casos, o PBIA simplesmente nunca usa aquela palavra.

O padrão que aparece é nítido: o vocabulário mais típico do PBIA é sobre
**pessoas e entrega de serviços** — saúde, qualificação profissional,
redução (de desigualdades, de custos, de dependência externa — o termo
aparece em vários contextos sociais e de gestão pública), evasão escolar,
população. É a linguagem de quem
está descrevendo programas concretos para atender gente. Já o vocabulário
mais típico dos outros sete documentos é sobre **indústria e tecnologia em
escala** — fábricas, manufatura, aceleração, implantação, avanços
revolucionários. É a linguagem de quem está descrevendo a construção de
capacidade tecnológica e industrial.

Nenhuma das duas linguagens é "melhor" — são focos diferentes. Mas o
contraste ajuda a entender concretamente **por que** a Seção 6 (perfil
temático) encontra o PBIA num meio-termo entre o registro de direitos/
desenvolvimento social e o registro de infraestrutura técnica dos outros
documentos: aqui dá para ver, palavra por palavra, de onde vem essa
diferença.

*(Para os números completos — estatística G², contagens exatas, todos os
116 termos significativos — veja o gráfico técnico logo acima.)*

---

# 6. Preferência temática — direitos humanos, desenvolvimento nacional ou outro foco?

## Metodologia (léxico deduzido a priori — limitação declarada)

Construímos 5 categorias temáticas por **análise de conteúdo baseada em
dicionário** (abordagem clássica, no espírito do LIWC): uma lista de
palavras-raiz definida **a priori** por nós, não extraída estatisticamente
dos dados. Isso é uma limitação metodológica real e deve ser lida como tal:
o léxico reflete escolhas do pesquisador sobre o que conta como "vocabulário
de direitos" ou "vocabulário de desenvolvimento" — outro pesquisador
poderia construir listas um pouco diferentes. Cada palavra do léxico foi
convertida para sua forma normalizada (stem) usando o mesmo stemmer da
Seção 2, e as listas completas estão no código abaixo — nenhuma é uma caixa
preta.

Cada raiz aparece em **no máximo uma** categoria (verificado por checagem
automática de sobreposição no código). Para cada documento, calculamos a
**participação relativa de cada categoria no total de tokens
categorizados** daquele documento (soma 100% por documento) — não a
frequência absoluta, porque documentos maiores teriam mais menções em
termos absolutos de qualquer categoria só por serem maiores.

In [ ]:
THEME_WORDS = {
    "Direitos, ética e governança responsável": [
        "rights", "right", "human", "dignity", "fairness", "fair", "discrimination", "discriminatory",
        "privacy", "bias", "biased", "ethics", "ethical", "accountability", "accountable", "transparency",
        "transparent", "inclusion", "inclusive", "equity", "equitable", "trust", "trustworthy", "responsible",
        "responsibility", "vulnerable", "protection", "protect", "consent", "diversity", "diverse", "harm",
        "safety", "safe",
    ],
    "Desenvolvimento nacional e econômico": [
        "development", "developing", "economic", "economy", "growth", "industry", "industrial",
        "competitiveness", "competitive", "gdp", "jobs", "job", "employment", "investment", "investing",
        "productivity", "product", "market", "trade", "capital", "prosperity",
    ],
    "Segurança e geopolítica": [
        "security", "secure", "sovereignty", "sovereign", "defense", "defence", "threat", "threats", "risk",
        "risks", "race", "leadership", "dominance", "dominant", "geopolitical", "military",
    ],
    "Governança, regulação e institucionalidade": [
        "governance", "government", "regulation", "regulatory", "policy", "policies", "law", "laws", "legal",
        "standards", "standard", "compliance", "oversight", "framework", "frameworks", "institutional",
        "institution", "agency", "agencies", "council", "board",
    ],
    "Infraestrutura e capacidade técnica": [
        "infrastructure", "computing", "computational", "data", "chips", "chip", "hardware", "cloud", "energy",
        "talent", "skills", "skill", "training", "research", "capacity", "semiconductor", "semiconductors",
        "supercomputer", "supercomputers",
    ],
}
THEME_STEMS = {cat: set(stem(w) for w in words) for cat, words in THEME_WORDS.items()}
THEME_COLORS = {
    "Direitos, ética e governança responsável": "#e34948",
    "Desenvolvimento nacional e econômico": "#008300",
    "Segurança e geopolítica": "#1a3d7c",
    "Governança, regulação e institucionalidade": "#eda100",
    "Infraestrutura e capacidade técnica": "#4a3aa7",
}

# checagem automática: nenhuma raiz deve pertencer a duas categorias ao mesmo tempo
seen = {}
for cat, stems_set in THEME_STEMS.items():
    for s in stems_set:
        assert s not in seen, f"stem \'{s}\' em duas categorias: {seen.get(s)} e {cat}"
        seen[s] = cat
print(f"OK: {len(seen)} raízes distribuídas em {len(THEME_STEMS)} categorias, sem sobreposição.")

theme_profile = {}
for fn in doc_order:
    counts = counts_new[fn]
    cat_counts = {cat: sum(counts.get(s, 0) for s in stems_set) for cat, stems_set in THEME_STEMS.items()}
    total_cat = sum(cat_counts.values())
    theme_profile[fn] = cat_counts, total_cat

for fn in doc_order:
    cat_counts, total_cat = theme_profile[fn]
    print(f"\n{LABELS[fn]} — {total_cat} tokens temáticos de {len(tokens_new[fn])} totais ({total_cat/len(tokens_new[fn])*100:.1f}%)")
    for cat, c in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"    {cat:45s} {c/total_cat*100:5.1f}%")

### Gráfico — perfil temático de cada documento

Barras empilhadas horizontais, uma por documento, sempre somando 100% —
mostra a composição temática relativa de cada um, permitindo ver de
imediato se o PBIA se parece mais com o perfil de algum documento
específico ou tem um perfil próprio.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

cats_ordered = list(THEME_WORDS.keys())
plot_order = doc_order[::-1]  # PBIA no topo
y_pos = range(len(plot_order))

left = [0.0] * len(plot_order)
for cat in cats_ordered:
    widths = []
    for fn in plot_order:
        cat_counts, total_cat = theme_profile[fn]
        widths.append(cat_counts[cat] / total_cat * 100 if total_cat else 0.0)
    ax.barh(list(y_pos), widths, left=left, color=THEME_COLORS[cat], height=0.62, zorder=3,
            label=cat, edgecolor=SURFACE, linewidth=1.2)
    for y, w, l in zip(y_pos, widths, left):
        if w > 5:
            ax.text(l + w / 2, y, f"{w:.0f}%", va="center", ha="center", fontsize=7.7,
                    color="white", fontweight="bold")
    left = [l + w for l, w in zip(left, widths)]

ax.set_yticks(list(y_pos))
labels_bold = [f"PBIA (Brasil)" if fn == "PBIA.json" else LABELS[fn] for fn in plot_order]
ax.set_yticklabels(labels_bold, color=INK_PRIMARY, fontsize=10)
for tick, fn in zip(ax.get_yticklabels(), plot_order):
    if fn == "PBIA.json":
        tick.set_fontweight("bold")
ax.set_xlim(0, 100)
ax.set_xlabel("% do vocabulário temático categorizado do documento", color=INK_MUTED, fontsize=9.5)
ax.set_title("Perfil temático — em que o PBIA e cada documento internacional mais falam?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False,
          labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()

In [ ]:
# Complementa a similaridade de vocabulário (Seção 3, cosseno sobre termos) com a
# similaridade de FORMATO do perfil temático (distância euclidiana entre os vetores
# de % por categoria) — pergunta diferente: não "os documentos usam as mesmas
# palavras", mas "os documentos distribuem sua ênfase pelas 5 categorias do mesmo
# jeito proporcional".
pbia_shares = {cat: theme_profile["PBIA.json"][0][cat] / theme_profile["PBIA.json"][1] * 100 for cat in cats_ordered}

shape_dist = {}
for fn in ref_order:
    cat_counts, total_cat = theme_profile[fn]
    shares = {cat: cat_counts[cat] / total_cat * 100 for cat in cats_ordered}
    dist = math.sqrt(sum((pbia_shares[cat] - shares[cat]) ** 2 for cat in cats_ordered))
    shape_dist[fn] = dist

print("Distância do FORMATO do perfil temático do PBIA a cada documento (pontos percentuais, menor = perfil mais parecido):")
for fn in sorted(ref_order, key=lambda fn: shape_dist[fn]):
    print(f"  {LABELS[fn]:32s} {shape_dist[fn]:5.1f} p.p.")

---

# 7. Mapa de posicionamento consolidado — onde o PBIA se situa entre os 7

Escalonamento multidimensional clássico (MDS) sobre a matriz completa de
distâncias (1 − similaridade de cosseno) entre os 8 documentos — projeta o
espaço de similaridade textual em 2D preservando ao máximo as distâncias
originais. Mesma técnica já validada no `Comparações.ipynb`, agora sobre o
corpus corrigido (Seções 1–2).

In [ ]:
n = len(doc_order)
S = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        S[i, j] = 1.0 if i == j else cosine_sim(
            counts_new[doc_order[i]], len(tokens_new[doc_order[i]]),
            counts_new[doc_order[j]], len(tokens_new[doc_order[j]]))
D = 1.0 - S
np.fill_diagonal(D, 0.0)


def classical_mds(dist_matrix, n_components=2):
    n = dist_matrix.shape[0]
    D2 = dist_matrix ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    eigvals_pos = np.clip(eigvals, 0, None)
    coords = eigvecs[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    explained = eigvals_pos[:n_components].sum() / eigvals_pos.sum()
    return coords, explained


coords, explained = classical_mds(D)
print(f"Variância explicada pelos 2 componentes: {explained*100:.1f}%")
for label, (x, y) in zip(doc_order, coords):
    print(f"  {LABELS[label]:32s} ({x:+.3f}, {y:+.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for i in range(n):
    for j in range(i + 1, n):
        sim = S[i, j]
        ax.plot([coords[i, 0], coords[j, 0]], [coords[i, 1], coords[j, 1]],
                color=INK_MUTED, linewidth=0.5 + sim * 3.0, alpha=0.15 + sim * 0.35, zorder=1)

LABEL_OFFSETS = {
    "PBIA.json": (-14, -22),
    "BRICS.json": (0, 16),
    "OCDE.json": (0, 16),
    "AI_PLUS.json": (-8, -20),
    "China_New_Gen.json": (0, 16),
    "Apply_AI_Strategy.json": (14, -20),
    "AI_CONTINENT_ACTION_PLAN.json": (10, 18),
    "AMERICA_AI_ACTION_PLAN.json": (18, 14),
}

for i, fn in enumerate(doc_order):
    is_pbia = fn == "PBIA.json"
    ax.scatter([coords[i, 0]], [coords[i, 1]], s=480 if is_pbia else 320, color=COLOR[fn],
               alpha=0.95, edgecolors=INK_PRIMARY if is_pbia else SURFACE,
               linewidths=1.6 if is_pbia else 1.2, zorder=4, marker="D" if is_pbia else "o")
    dx, dy = LABEL_OFFSETS[fn]
    use_arrow = abs(dx) + abs(dy) > 24
    ax.annotate(LABELS[fn], (coords[i, 0], coords[i, 1]), textcoords="offset points",
                xytext=(dx, dy), ha="center", fontsize=10 if is_pbia else 9.5,
                fontweight="bold" if is_pbia else "normal", color=INK_PRIMARY, zorder=5,
                arrowprops=dict(arrowstyle="-", color=INK_MUTED, linewidth=0.8, shrinkA=2, shrinkB=8)
                if use_arrow else None)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
pad = max(np.abs(coords).max() * 0.5, 0.05)
ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)
ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(f"Mapa de similaridade textual dos 8 documentos (MDS, {explained*100:.0f}% da variância)",
             color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16)
for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

### 7.1. Mapa complementar — sem BRICS e sem OCDE (maior fidelidade geométrica)

O mapa da Seção 7 tem um problema documentado: o **stress de Kruskal do
ajuste é 0,291** (limiares convencionais: <0,05 excelente, 0,05–0,1 bom,
0,1–0,2 razoável, >0,2 ruim — 0,291 está em território ruim), e a
distância PBIA–America's AI Action Plan aparece comprimida para **11% da
distância real** (1 − cosseno) — a maior distorção de qualquer par do
mapa. A causa identificável: BRICS e OCDE são os dois únicos documentos do
conjunto **sem um par muito coeso** (diferente de AI+↔China New Gen,
cosseno 0,796, e Apply↔AI Continent, cosseno 0,778) — sua presença força
uma configuração geométrica que distorce as posições dos documentos
"generalistas" (PBIA e America's AI Action Plan) para acomodar esses dois
pontos mais extremos.

Este segundo mapa recalcula o MDS **apenas com os cinco documentos
nacionais/de bloco de IA + o PBIA** (sem BRICS nem OCDE, que são
declarações multilaterais de princípios, não planos nacionais). Ele não
substitui o mapa da Seção 7 — **não mostra onde o PBIA fica em relação ao
seu par mais próximo (BRICS, cosseno 0,649) nem ao mais distante (OCDE,
0,462)**, que são as duas conclusões mais robustas do estudo (Seção 8.2).
É um mapa complementar, de leitura mais confiável especificamente para a
posição do PBIA **entre as políticas nacionais de IA**, ao custo de deixar
de fora os dois documentos multilaterais de referência.

In [ ]:
doc_order_6 = [fn for fn in doc_order if fn not in ("BRICS.json", "OCDE.json")]
n6 = len(doc_order_6)
S6 = np.zeros((n6, n6))
for i in range(n6):
    for j in range(n6):
        S6[i, j] = 1.0 if i == j else cosine_sim(
            counts_new[doc_order_6[i]], len(tokens_new[doc_order_6[i]]),
            counts_new[doc_order_6[j]], len(tokens_new[doc_order_6[j]]))
D6 = 1.0 - S6
np.fill_diagonal(D6, 0.0)

coords6, explained6 = classical_mds(D6)

from itertools import combinations
num, den = 0.0, 0.0
for i, j in combinations(range(n6), 2):
    real_d = D6[i, j]
    mds_d = np.linalg.norm(coords6[i] - coords6[j])
    num += (real_d - mds_d) ** 2
    den += real_d ** 2
stress6 = math.sqrt(num / den)

print(f"Variância explicada pelos 2 componentes: {explained6*100:.1f}%  (era 67,0% com 8 documentos)")
print(f"Stress de Kruskal: {stress6:.3f}  (era 0,291 com 8 documentos)")
print()

pbia_i6 = doc_order_6.index("PBIA.json")
print("Fidelidade da distância do PBIA a cada documento (real vs. mapa 2D):")
for j in range(n6):
    if j == pbia_i6:
        continue
    real_d = D6[pbia_i6, j]
    mds_d = np.linalg.norm(coords6[pbia_i6] - coords6[j])
    print(f"  PBIA - {LABELS[doc_order_6[j]]:32s} real={real_d:.3f}  mapa2D={mds_d:.3f}  razão={mds_d/real_d:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for i in range(n6):
    for j in range(i + 1, n6):
        sim = S6[i, j]
        ax.plot([coords6[i, 0], coords6[j, 0]], [coords6[i, 1], coords6[j, 1]],
                color=INK_MUTED, linewidth=0.5 + sim * 3.0, alpha=0.15 + sim * 0.35, zorder=1)

LABEL_OFFSETS_6 = {
    "PBIA.json": (-16, -20),
    "AI_PLUS.json": (0, -18),
    "China_New_Gen.json": (0, 16),
    "Apply_AI_Strategy.json": (0, -18),
    "AI_CONTINENT_ACTION_PLAN.json": (14, 16),
    "AMERICA_AI_ACTION_PLAN.json": (0, 16),
}

for i, fn in enumerate(doc_order_6):
    is_pbia = fn == "PBIA.json"
    ax.scatter([coords6[i, 0]], [coords6[i, 1]], s=480 if is_pbia else 320, color=COLOR[fn],
               alpha=0.95, edgecolors=INK_PRIMARY if is_pbia else SURFACE,
               linewidths=1.6 if is_pbia else 1.2, zorder=4, marker="D" if is_pbia else "o")
    dx, dy = LABEL_OFFSETS_6[fn]
    ax.annotate(LABELS[fn], (coords6[i, 0], coords6[i, 1]), textcoords="offset points",
                xytext=(dx, dy), ha="center", fontsize=10 if is_pbia else 9.5,
                fontweight="bold" if is_pbia else "normal", color=INK_PRIMARY, zorder=5)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
pad = max(np.abs(coords6).max() * 0.5, 0.05)
ax.set_xlim(coords6[:, 0].min() - pad, coords6[:, 0].max() + pad)
ax.set_ylim(coords6[:, 1].min() - pad, coords6[:, 1].max() + pad)
ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"PBIA entre as 5 políticas nacionais/de bloco de IA — sem BRICS/OCDE\n"
    f"(MDS, {explained6*100:.0f}% da variância, stress de Kruskal {stress6:.3f})",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16)
for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

### Versão para publicação (artigo)

A célula acima é a versão técnica, com eixos numéricos e legenda estatística
completa. A célula abaixo apresenta **exatamente os mesmos pontos, as
mesmas coordenadas e a mesma matriz de similaridade** — nenhum número foi
recalculado — só com uma leitura visual pensada para publicação: título sem
jargão, e os eixos nomeados a partir do vocabulário que estatisticamente
mais se associa a cada direção (técnica de *vector fitting*, detalhada na
caixa de texto logo abaixo do gráfico).

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 9.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Mesmas linhas de conexão (espessura proporcional à similaridade) e os mesmos
# pontos/cores/rótulos de documento da célula técnica acima — nada recalculado aqui.
for i in range(n6):
    for j in range(i + 1, n6):
        sim = S6[i, j]
        ax.plot([coords6[i, 0], coords6[j, 0]], [coords6[i, 1], coords6[j, 1]],
                color=INK_MUTED, linewidth=0.5 + sim * 3.0, alpha=0.15 + sim * 0.35, zorder=1)

for i, fn in enumerate(doc_order_6):
    is_pbia = fn == "PBIA.json"
    ax.scatter([coords6[i, 0]], [coords6[i, 1]], s=560 if is_pbia else 380, color=COLOR[fn],
               alpha=0.95, edgecolors=INK_PRIMARY if is_pbia else SURFACE,
               linewidths=1.8 if is_pbia else 1.3, zorder=4, marker="D" if is_pbia else "o")
    dx, dy = LABEL_OFFSETS_6[fn]
    ax.annotate(LABELS[fn], (coords6[i, 0], coords6[i, 1]), textcoords="offset points",
                xytext=(dx, dy), ha="center", fontsize=11.5 if is_pbia else 10.5,
                fontweight="bold" if is_pbia else "normal", color=INK_PRIMARY, zorder=5)

ax.axhline(0, color=GRIDLINE, linewidth=1.2, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=1.2, zorder=0)

pad = max(np.abs(coords6).max() * 0.62, 0.05)
xlim = (coords6[:, 0].min() - pad, coords6[:, 0].max() + pad)
ylim = (coords6[:, 1].min() - pad * 1.45, coords6[:, 1].max() + pad * 1.45)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# Rótulos direcionais dos quatro polos de vocabulário, no lugar dos eixos numéricos
# (fundamentados na correlação termo-eixo — ver caixa de texto técnica abaixo)
ax.text(xlim[0] * 0.97, 0.014, "← Vocabulário China\n(arquitetura técnica de IA)",
        ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=COLOR["China_New_Gen.json"])
ax.text(xlim[1] * 0.97, 0.014, "Vocabulário UE →\n(regulatório-institucional)",
        ha="right", va="bottom", fontsize=10.5, fontweight="bold", color=COLOR["AI_CONTINENT_ACTION_PLAN.json"])
ax.text(0, ylim[1] * 0.96, "↑ Vocabulário EUA\n(competição geopolítica e administração federal)",
        ha="center", va="top", fontsize=10.5, fontweight="bold", color=COLOR["AMERICA_AI_ACTION_PLAN.json"])
ax.text(0, ylim[0] * 0.96, "↓ Vocabulário comum\n(suporte e implementação técnica)",
        ha="center", va="bottom", fontsize=10.5, fontweight="bold", color=INK_MUTED)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

fig.subplots_adjust(top=0.84)
fig.suptitle("Proximidade de linguagem entre as principais políticas\nnacionais de inteligência artificial",
             color=INK_PRIMARY, fontsize=16, fontweight="bold", y=0.985, linespacing=1.35)
fig.text(0.5, 0.875, "PBIA (Brasil), China, União Europeia e Estados Unidos",
         ha="center", fontsize=11.5, color=INK_MUTED, style="italic")

plt.show()

### Nota técnica sobre o gráfico acima

Esta versão foi simplificada para leitura em publicação — esta caixa
documenta o que foi omitido do desenho, para que o método continue
totalmente rastreável.

**Como o mapa é calculado.** Cada um dos 6 documentos vira um vetor de
frequência relativa (ocorrências por 1.000 tokens) sobre o vocabulário
combinado. A partir daí calculamos a similaridade de cosseno entre cada
par (15 pares) e convertemos em distância (1 − cosseno). Um algoritmo de
escalonamento multidimensional clássico (MDS, método Torgerson-Gower)
decompõe essa matriz de distâncias em autovalores/autovetores e projeta os
6 documentos num plano 2D que preserva o máximo possível das 15 distâncias
**simultaneamente** — não é um ajuste ponto a ponto, é a melhor solução de
compromisso para o conjunto todo.

**O que os eixos significam — e por que os números somem nesta versão.**
Os valores brutos de X e Y (visíveis na célula técnica logo acima) não têm
unidade interpretável: são coordenadas sintéticas derivadas dos
autovalores, cuja escala não corresponde a nenhuma grandeza textual direta
(não é "ocorrências por mil palavras" nem nada parecido). Só a **distância
relativa** entre dois pontos tem significado. Por isso, nesta versão para
publicação, os números do eixo foram removidos — mantê-los sugeriria uma
precisão numérica que os valores não têm.

**De onde vêm os quatro rótulos de vocabulário.** Não foram escolhidos por
inspeção visual. Calculamos a correlação de Pearson entre a frequência de
cada termo do vocabulário (nos 6 documentos) e a coordenada de cada
documento em X e em Y — técnica conhecida como *vector fitting*, usada em
ecologia/estatística multivariada para interpretar eixos de ordenação.
Filtrando por termos com frequência total ≥ 50 (para reduzir ruído):

| Eixo | Extremo | Termos mais correlacionados |
|---|---|---|
| X negativo | China | *intelligent/intelligence* (r=-0,92; 331 ocorrências), *develop* (r=-0,89; 537 ocorrências) |
| X positivo | UE | *strategy* (r=+0,95; 194 ocorrências), *market* (r=+0,95; 69 ocorrências) |
| Y positivo | EUA | *federal* (r=+0,99; 52 ocorrências), *agency* (r=+0,97; 35 ocorrências) |
| Y negativo | comum UE+China | *support* (r=-0,96; 227 ocorrências), *implementation* (r=-0,82; 54 ocorrências) |

**Ressalvas importantes para quem for analisar ou citar este gráfico:**

1. **A rotulagem dos eixos é uma leitura *a posteriori*, não uma escala
   desenhada previamente.** Ela é válida para esta configuração específica
   de 6 documentos; adicionar ou remover um documento pode reorientar os
   eixos e exigir recalcular as correlações.
2. **A correlação usada para nomear os eixos tem só 6 observações** (um
   ponto por documento) — com graus de liberdade = 4, o r crítico para
   p<0,05 é ≈0,81. Os valores reportados na tabela são bem acima disso e
   vêm de termos com frequência razoável, mas o tamanho de amostra
   continua pequeno; trate os rótulos como uma caracterização qualitativa
   fundamentada, não como coeficientes estatísticos definitivos.
3. **O mapa 2D preserva cerca de 80% da estrutura real das distâncias**
   (autovalores das dimensões 3, 4 e 5 são descartados — Seção 7.2.1); o
   *stress* de Kruskal do ajuste é 0,206, o que pelos limiares
   convencionais (Kruskal, 1964) ainda está em território "ruim" (>0,2),
   mesmo sendo bem melhor que a versão com 8 documentos (0,291).
4. **As distâncias no mapa são comprimidas de forma relativamente uniforme**
   a 65–78% da distância real (tabela de fidelidade par a par na Seção
   7.1) — a posição relativa dos pontos é informativa, mas não deve ser
   lida como medida quantitativa precisa.
5. Esta figura e a célula técnica anterior usam **exatamente os mesmos
   dados e o mesmo cálculo** — para os valores numéricos de cada eixo, a
   matriz de similaridade completa e o diagnóstico de autovalores, veja a
   Seção 7.1 (célula técnica) e a Seção 7.2.1.

### Versão híbrida — nomes intuitivos com os valores numéricos visíveis

Terceira variação, para comparação: o mesmo visual da versão para
publicação (rótulos de vocabulário nos eixos), mas com os valores
numéricos dos eixos de volta — para quem preferir ver a escala exata por
trás de cada posição junto com a leitura facilitada. Os rótulos de
vocabulário do eixo X foram reposicionados para fora da área do gráfico
(abaixo dos números), para não ficarem cortados pela linha do eixo.

In [ ]:
from matplotlib.ticker import MultipleLocator
fig, ax = plt.subplots(figsize=(9.5, 9.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for i in range(n6):
    for j in range(i + 1, n6):
        sim = S6[i, j]
        ax.plot([coords6[i, 0], coords6[j, 0]], [coords6[i, 1], coords6[j, 1]],
                color=INK_MUTED, linewidth=0.5 + sim * 3.0, alpha=0.15 + sim * 0.35, zorder=1)

for i, fn in enumerate(doc_order_6):
    is_pbia = fn == "PBIA.json"
    ax.scatter([coords6[i, 0]], [coords6[i, 1]], s=560 if is_pbia else 380, color=COLOR[fn],
               alpha=0.95, edgecolors=INK_PRIMARY if is_pbia else SURFACE,
               linewidths=1.8 if is_pbia else 1.3, zorder=4, marker="D" if is_pbia else "o")
    dx, dy = LABEL_OFFSETS_6[fn]
    ax.annotate(LABELS[fn], (coords6[i, 0], coords6[i, 1]), textcoords="offset points",
                xytext=(dx, dy), ha="center", fontsize=11.5 if is_pbia else 10.5,
                fontweight="bold" if is_pbia else "normal", color=INK_PRIMARY, zorder=5)

ax.axhline(0, color=GRIDLINE, linewidth=1.2, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=1.2, zorder=0)

pad = max(np.abs(coords6).max() * 0.62, 0.05)
xlim = (coords6[:, 0].min() - pad, coords6[:, 0].max() + pad)
ylim = (coords6[:, 1].min() - pad * 1.45, coords6[:, 1].max() + pad * 1.45)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# Eixo Y: rótulos de vocabulário mantidos dentro da área do gráfico (topo/base),
# longe o bastante do y=0 e dos pontos para não colidir com nada.
ax.text(0, ylim[1] * 0.96, "↑ Vocabulário EUA\n(competição geopolítica e administração federal)",
        ha="center", va="top", fontsize=10, fontweight="bold", color=COLOR["AMERICA_AI_ACTION_PLAN.json"])
ax.text(0, ylim[0] * 0.90, "↓ Vocabulário comum\n(suporte e implementação técnica)",
        ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK_MUTED)

# Eixo X: rótulos de vocabulário sobre a linha y=0, em duas linhas, ancorados para
# crescer para CIMA a partir de y=0 (todos os documentos China/UE estão abaixo de
# y=0, então isso evita colidir com os rótulos "China New Gen"/"AI Continent...").
# Um retângulo de fundo na cor do papel (bbox) garante que a linha do eixo não
# corte visualmente o texto.
ax.text(xlim[0] * 0.97, 0.09, "← Vocabulário China\n(arquitetura técnica de IA)",
        ha="left", va="bottom", fontsize=10, fontweight="bold",
        color=COLOR["China_New_Gen.json"], zorder=3)
ax.text(xlim[1] * 0.97, 0.09, "Vocabulário UE →\n(regulatório-institucional)",
        ha="right", va="bottom", fontsize=10, fontweight="bold",
        color=COLOR["AI_CONTINENT_ACTION_PLAN.json"], zorder=3)

ax.xaxis.set_major_locator(MultipleLocator(0.1))
ax.yaxis.set_major_locator(MultipleLocator(0.1))

ax.tick_params(colors=INK_MUTED, labelsize=9)
for spine in ax.spines.values():
    spine.set_color(GRIDLINE)

fig.subplots_adjust(top=0.84, bottom=0.14)
fig.suptitle("Proximidade de linguagem entre as principais políticas\nnacionais de inteligência artificial",
             color=INK_PRIMARY, fontsize=16, fontweight="bold", y=0.985, linespacing=1.35)
fig.text(0.5, 0.875, "PBIA (Brasil), China, União Europeia e Estados Unidos",
         ha="center", fontsize=11.5, color=INK_MUTED, style="italic")

plt.show()

---

# 7.2. Diagnóstico do mapa e o vocabulário por trás das posições

Esta seção documenta, com evidência computável, três coisas que qualquer
uso acadêmico do mapa da Seção 7.1 precisa poder citar: **(a)** se a
matriz de distâncias usada pelo MDS é estruturalmente bem-comportada,
**(b)** que vocabulário específico faz os documentos se agruparem do jeito
que aparecem no mapa, e **(c)** que vocabulário do PBIA especificamente o
puxa para cada lado.

### 7.2.1. A matriz de distâncias cabe num espaço euclidiano?

O MDS clássico decompõe a matriz de distâncias em autovalores. Se algum
autovalor relevante for **negativo**, isso significa que a matriz de
distâncias (aqui, 1 − cosseno) não se encaixa perfeitamente em nenhum
espaço euclidiano — um problema estrutural mais sério do que a simples
perda de variância ao projetar em 2D, porque indicaria que a própria noção
de "distância" entre os documentos, do jeito que foi calculada, é
internamente inconsistente. Isso nunca tinha sido checado explicitamente
neste notebook.

In [ ]:
# Reaproveita D6 e n6 já calculados na Seção 7.1 — não recalcula a matriz de
# similaridade, só decompõe a mesma matriz de distâncias em autovalores.
D2_6 = D6 ** 2
J6 = np.eye(n6) - np.ones((n6, n6)) / n6
B6 = -0.5 * J6 @ D2_6 @ J6
eigvals6 = np.sort(np.linalg.eigvalsh(B6))[::-1]

print("Autovalores da matriz de dupla-centralização (n-1 = 5 dimensões não-triviais):")
for i, ev in enumerate(eigvals6):
    print(f"  dimensão {i + 1}: {ev:+.5f}")

total_pos = sum(e for e in eigvals6 if e > 0)
total_neg = sum(e for e in eigvals6 if e < 0)
print(f"\nSoma dos autovalores positivos: {total_pos:.5f}")
print(f"Soma dos autovalores negativos: {total_neg:.5f}")

if abs(total_neg) < 1e-6:
    print("\nNenhum autovalor negativo relevante -> a matriz de distância É perfeitamente")
    print("encaixável em espaço euclidiano. O mapa em 2D perde informação real (as")
    print("dimensões 3, 4 e 5 são descartadas), mas não há distorção estrutural escondida.")
else:
    print(f"\nAutovalores negativos somam {abs(total_neg) / total_pos * 100:.1f}% dos positivos")
    print("-> a matriz de distância NÃO se encaixa perfeitamente em nenhum espaço euclidiano.")

descartada = (1 - (eigvals6[0] + eigvals6[1]) / total_pos) * 100
print(f"\nVariância real descartada ao usar só as 2 dimensões do mapa: {descartada:.1f}%")
print(f"(consistente com os {explained6*100:.1f}% de variância explicada já reportados na Seção 7.1,")
print(f"e com o stress de Kruskal de {stress6:.3f} — a distorção vem inteiramente de comprimir")
print("5 dimensões reais em 2, não de nenhum problema na matriz de distância em si.)")

### 7.2.2. O vocabulário por trás dos quatro polos do mapa

A matriz de similaridade completa (Seção 7.1) mostra dois pares muito mais
coesos entre si do que com qualquer outro documento: **AI+ × China New
Gen = 0,796** e **Apply AI Strategy × AI Continent Action Plan = 0,778** —
ambos bem acima de qualquer similaridade envolvendo o PBIA (0,57–0,64). É
esse contraste que empurra os dois pares para cantos opostos do mapa e
deixa o PBIA e o America's AI Action Plan — que não têm um par tão coeso —
mais perto do centro.

Para comparar os **quatro polos do mapa em pé de igualdade** — cluster
China, cluster UE, America's AI Action Plan e PBIA —, aplicamos o mesmo
teste de log-likelihood (G²) já usado na Seção 5 quatro vezes: nos dois
clusters, o "grupo" é o par de documentos combinado; no America's AI
Action Plan e no PBIA, o "grupo" é o próprio documento sozinho (nenhum dos
dois tem um par coeso para combinar). Em todos os quatro casos, o "resto"
é a soma dos outros quatro documentos do mapa de 6.

In [ ]:
def top_distinctive_group(group_docs, rest_docs, label, n=10):
    gc, gt = Counter(), 0
    for fn in group_docs:
        gc += counts_new[fn]
        gt += len(tokens_new[fn])
    rc, rt = Counter(), 0
    for fn in rest_docs:
        rc += counts_new[fn]
        rt += len(tokens_new[fn])
    rows = []
    for t in set(gc) | set(rc):
        if t in SELF_REFERENTIAL_KEYNESS:
            continue
        a, b = gc.get(t, 0), rc.get(t, 0)
        if a + b < MIN_COUNT_KEYNESS:
            continue
        g2 = log_likelihood(a, gt, b, rt)
        if (a / gt) <= (b / rt):  # só termos que puxam PARA o grupo, não para o resto
            continue
        rows.append((t, a, b, g2, a / gt * 1000, b / rt * 1000))
    rows.sort(key=lambda r: -r[3])
    print(f"Vocabulário que caracteriza {label}:")
    for r in rows[:n]:
        print(f"  {r[0]:14s} G²={r[3]:7.1f}  grupo={r[1]:3d} ({r[4]:5.2f}‰)  resto={r[2]:3d} ({r[5]:5.2f}‰)")
    print()
    return rows[:n]


china_cluster_docs = ["AI_PLUS.json", "China_New_Gen.json"]
eu_cluster_docs = ["Apply_AI_Strategy.json", "AI_CONTINENT_ACTION_PLAN.json"]
eua_docs = ["AMERICA_AI_ACTION_PLAN.json"]
pbia_docs_solo = ["PBIA.json"]

china_cluster_vocab = top_distinctive_group(
    china_cluster_docs, eu_cluster_docs + eua_docs + pbia_docs_solo,
    "AI+ / China New Gen (cluster China, cosseno 0,796 entre si)")
eu_cluster_vocab = top_distinctive_group(
    eu_cluster_docs, china_cluster_docs + eua_docs + pbia_docs_solo,
    "Apply AI Strategy / AI Continent (cluster UE, cosseno 0,778 entre si)")
eua_vocab = top_distinctive_group(
    eua_docs, china_cluster_docs + eu_cluster_docs + pbia_docs_solo,
    "America's AI Action Plan (documento único, sem par coeso)")
pbia_vocab = top_distinctive_group(
    pbia_docs_solo, china_cluster_docs + eu_cluster_docs + eua_docs,
    "PBIA (documento único, sem par coeso)")

In [ ]:
# Algumas raízes do stemmer não são palavras completas (ex.: "intelligen",
# "technolog") — trocamos por um rótulo legível só para exibição no gráfico,
# sem alterar o dado (o cálculo por trás continua no stem original).
STEM_DISPLAY_LABEL = {
    "intelligen": "intelligent/-ce", "technolog": "technology", "strateg": "strategy/-ic",
    "appl": "application", "strength": "strengthen", "artifici": "artificial",
    "creat": "create/-ion", "glob": "global", "improv": "improve/-ment", "capac": "capacity",
    "govern": "govern/-ance", "feder": "federal", "qualif": "qualification", "increas": "increase",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 12), facecolor=SURFACE)

panel_specs = [
    (axes[0, 0], china_cluster_vocab, COLOR["China_New_Gen.json"], "Cluster China\n(AI+ e China New Gen)", "no cluster"),
    (axes[0, 1], eu_cluster_vocab, COLOR["AI_CONTINENT_ACTION_PLAN.json"], "Cluster UE\n(Apply AI Strategy e AI Continent)", "no cluster"),
    (axes[1, 0], eua_vocab, COLOR["AMERICA_AI_ACTION_PLAN.json"], "America's AI Action Plan\n(documento único)", "no documento"),
    (axes[1, 1], pbia_vocab, COLOR["PBIA.json"], "PBIA\n(documento único)", "no documento"),
]

for ax, vocab, color, title, group_legend in panel_specs:
    ax.set_facecolor(SURFACE)
    rows = vocab[:8][::-1]
    labels = [STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in rows]
    group_rates = [r[4] for r in rows]
    rest_rates = [r[5] for r in rows]
    y_pos = np.arange(len(rows))
    bar_h = 0.35
    ax.barh(y_pos + bar_h / 2, group_rates, height=bar_h, color=color, zorder=3, label=group_legend)
    ax.barh(y_pos - bar_h / 2, rest_rates, height=bar_h, color=INK_MUTED, zorder=3, label="no resto (4 docs)")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=10)
    ax.set_xlabel("Frequência relativa (‰)", color=INK_MUTED, fontsize=9.5)
    ax.set_title(title, color=INK_PRIMARY, fontsize=11.5, fontweight="bold", pad=10)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.suptitle("Vocabulário estatisticamente responsável por cada um dos quatro polos do mapa",
             color=INK_PRIMARY, fontsize=14.5, fontweight="bold", y=1.0)
fig.tight_layout()
plt.show()

### 7.2.3. Que vocabulário do PBIA puxa para cada lado

Classificamos os termos mais usados pelo PBIA por qual dos três lados —
**bloco China** (AI+ + China New Gen combinados), **bloco UE** (Apply AI
Strategy + AI Continent combinados) ou **America's AI Action Plan** — tem
a maior frequência relativa daquele termo. Isso estende, para os cinco
documentos deste mapa (sem BRICS/OCDE), a mesma lógica já usada na Seção
5.

**Correção adicionada aqui em relação a versões anteriores desta análise**:
o lado "vencedor" só é aceito se tiver **pelo menos 3 ocorrências
absolutas** no bloco de referência — não basta ter frequência relativa
maior que zero. Sem esse piso, um termo com uma única ocorrência isolada
(ex.: 1 menção em um corpus pequeno) pode "vencer" por *default* contra
dois lados com zero ocorrências, produzindo uma classificação
estatisticamente frágil. Termos que não atingem o piso em nenhum lado
entram como **"evidência insuficiente"**, em vez de serem forçados a uma
categoria.

In [ ]:
MIN_COUNT_SIDE = 3     # piso mínimo de ocorrências absolutas no lado vencedor
MIN_PBIA_PULL = 4      # mínimo de ocorrências no PBIA para o termo entrar na análise
BALANCE_RATIO_PULL = 0.75

china_pull_counts = counts_new["AI_PLUS.json"] + counts_new["China_New_Gen.json"]
china_pull_total = len(tokens_new["AI_PLUS.json"]) + len(tokens_new["China_New_Gen.json"])
eu_pull_counts = counts_new["Apply_AI_Strategy.json"] + counts_new["AI_CONTINENT_ACTION_PLAN.json"]
eu_pull_total = len(tokens_new["Apply_AI_Strategy.json"]) + len(tokens_new["AI_CONTINENT_ACTION_PLAN.json"])
america_pull_counts = counts_new["AMERICA_AI_ACTION_PLAN.json"]
america_pull_total = len(tokens_new["AMERICA_AI_ACTION_PLAN.json"])

pull_rows = []
for t, c in pbia_counts.most_common():
    if t in SELF_REFERENTIAL_KEYNESS or c < MIN_PBIA_PULL:
        continue
    counts_by_side = {
        "China": china_pull_counts.get(t, 0),
        "UE": eu_pull_counts.get(t, 0),
        "EUA": america_pull_counts.get(t, 0),
    }
    rates_by_side = {
        "China": relative_freq(china_pull_counts, china_pull_total, t),
        "UE": relative_freq(eu_pull_counts, eu_pull_total, t),
        "EUA": relative_freq(america_pull_counts, america_pull_total, t),
    }
    if sum(rates_by_side.values()) == 0:
        continue
    ranked = sorted(rates_by_side.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_val = ranked[0]
    if counts_by_side[top_label] < MIN_COUNT_SIDE:
        group = "Evidência insuficiente"
    else:
        ratio = ranked[1][1] / top_val if top_val > 0 else 0.0
        group = "Equilibrado" if ratio >= BALANCE_RATIO_PULL else top_label
    pull_rows.append((t, c, rates_by_side["China"], rates_by_side["UE"], rates_by_side["EUA"], group))

pull_by_group = {"China": [], "UE": [], "EUA": [], "Equilibrado": [], "Evidência insuficiente": []}
for r in pull_rows:
    pull_by_group[r[5]].append(r)
for g in pull_by_group:
    pull_by_group[g].sort(key=lambda r: -r[1])

print(f"Total de termos do PBIA analisados: {len(pull_rows)}")
for g in ["China", "UE", "EUA", "Equilibrado", "Evidência insuficiente"]:
    print(f"  {g:24s}: {len(pull_by_group[g])} termos")
print()
for g in ["China", "UE", "EUA"]:
    print(f"Top termos que puxam para {g}:")
    for r in pull_by_group[g][:8]:
        print(f"  {r[0]:14s} PBIA={r[1]:3d}x  China={r[2]:6.2f}‰  UE={r[3]:6.2f}‰  EUA={r[4]:6.2f}‰")
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor=SURFACE)

panels = [
    (axes[0], "China", COLOR["China_New_Gen.json"], 2, "bloco China\n(AI+ + China New Gen)"),
    (axes[1], "UE", COLOR["AI_CONTINENT_ACTION_PLAN.json"], 3, "bloco UE\n(Apply + AI Continent)"),
    (axes[2], "EUA", COLOR["AMERICA_AI_ACTION_PLAN.json"], 4, "America's AI Action Plan"),
]

for ax, group, color, val_idx, title in panels:
    ax.set_facecolor(SURFACE)
    rows = pull_by_group[group][:8][::-1]
    labels = [STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in rows]
    values = [r[val_idx] for r in rows]
    pbia_n = [r[1] for r in rows]
    y_pos = range(len(rows))
    ax.barh(list(y_pos), values, color=color, height=0.62, zorder=3)
    for y, v, pc in zip(y_pos, values, pbia_n):
        ax.text(v + max(values) * 0.03, y, f"{v:.1f}‰ (PBIA:{pc}×)", va="center", ha="left",
                color=INK_PRIMARY, fontsize=8)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9.5)
    ax.set_xlim(0, max(values) * 1.5)
    ax.set_title(f"Puxam para o {title}", color=color, fontsize=10.5, fontweight="bold", pad=10)
    ax.set_xlabel("Frequência relativa no bloco (‰)", color=INK_MUTED, fontsize=8.5)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle("Vocabulário do PBIA que puxa para cada lado do mapa (piso mínimo: 3 ocorrências no lado vencedor)",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", y=1.03)
fig.tight_layout()
plt.show()

### Síntese do diagnóstico (7.2)

- **A matriz de distâncias é bem-comportada**: todos os autovalores
  não-triviais são positivos — a perda de precisão do mapa 2D vem inteira
  e apenas de descartar 3 das 5 dimensões reais, não de nenhuma
  inconsistência estrutural na forma como a distância foi calculada.
- **O cluster China** (AI+ e China New Gen, cosseno 0,796) é sustentado por
  vocabulário de **arquitetura técnica de IA** — *intelligent/intelligence*,
  *smart*, *system*, *platform*, *method* — o registro comum às duas
  políticas chinesas, apesar de seis anos de distância entre elas.
- **O cluster UE** (Apply AI Strategy e AI Continent, cosseno 0,778) é
  sustentado por vocabulário **regulatório-institucional** — *sector*,
  *strategy*, *member* (Estado-membro), *act* (como em "AI Act"), *SME* —
  incluindo termos exclusivos como *gigafactory*, nome de um programa
  europeu específico.
- **O America's AI Action Plan**, mesmo sozinho (sem par coeso), tem um
  vocabulário estatisticamente muito nítido: *federal*, *agency*,
  *adversary*, *export*, *ally*, *semiconductor* — o registro de
  **competição geopolítica e administração federal** ("vencer a corrida"
  contra um *adversary*, controlar *exports* de semicondutores, mobilizar
  *agencies* federais) que não aparece com essa intensidade em nenhum outro
  documento do mapa.
- **O PBIA**, também sozinho, confirma o mesmo padrão já visto na Seção 5
  mesmo com BRICS/OCDE fora da conta: *reduction*, *program*, *health*,
  *qualification* — vocabulário de **entrega social**, distinto tanto do
  registro técnico chinês quanto do regulatório europeu quanto do
  geopolítico americano.
- **O PBIA se divide de forma real entre os três lados**: dos termos
  analisados, um número substancial puxa para cada bloco (não dominado por
  nenhum), com um grupo à parte marcado como "evidência insuficiente" —
  precisamente os casos que, antes desta correção, poderiam ter sido
  classificados de forma frágil por uma única ocorrência isolada.

---

# 8. Síntese — como o PBIA se posiciona diante dos 7 documentos

## 8.1. Similaridade global de linguagem

Em similaridade de cosseno (ênfase, pondera frequência), o PBIA fica mais
próximo do **BRICS Declaration (0,649)** e mais distante da **OECD
Recommendation (0,462)** — a mesma amplitude, na direção, do
`Comparações.ipynb`. No índice de Jaccard (sobreposição pura de
vocabulário, sem pesar frequência), a ordem muda: **Apply AI Strategy
(0,311)** e **AI Continent Action Plan (0,300)** têm o maior número de
termos em comum com o PBIA, refletindo o vocabulário regulatório/setorial
que os três compartilham, mesmo com ênfases proporcionais diferentes.

## 8.2. O que é robusto e o que é sensível à metodologia (Seção 4)

Duas conclusões se sustentam **independentemente da extração de texto usada
ou do uso de stemming**: o **BRICS é sempre o documento mais próximo** e a
**OECD é sempre a mais distante**, nas duas metodologias. Já a ordem dos
cinco documentos do meio **não é robusta**: com o método antigo, os dois
documentos europeus apareciam em 2º/3º lugar; com a extração completa e
normalizada, o China New Gen e o America's AI Action Plan sobem para
2º/3º, e os dois documentos europeus caem para 5º/6º. A causa identificável
(Seção 4): a extração antiga do AI+ ignorava campos inteiros (metas de
desenvolvimento, iniciativas-chave), e a normalização morfológica beneficiou
desproporcionalmente o China New Gen e o America's AI Action Plan — ou
seja, parte do "PBIA está mais próximo da UE" do `Comparações.ipynb` era um
artefato de cobertura de texto incompleta, não um sinal real. Qualquer
leitura sobre *qual* documento específico ocupa o 2º ou 3º lugar deve ser
tratada como preliminar; que BRICS lidera e OCDE fica por último é sólido.

## 8.3. Vocabulário estatisticamente distintivo do PBIA (Seção 5)

Dos 1.189 termos testados (freq. combinada ≥ 5), **116 são estatisticamente
distintivos** (G² > 10,83, p < 0,001). O vocabulário que mais super-representa
o PBIA frente ao pool dos outros sete é de **execução social**: *program*
(62× vs. 23×), *reduction* (35× vs. 1×), *health* (52× vs. 23×),
*qualif[ication]* (23× vs. 0×), *structuring* (20× vs. 0×), *evasion* (12×
vs. 0×), *population* (21× vs. 7×) — termos ligados a programas concretos de
redução de desigualdade, evasão escolar, qualificação profissional e saúde
pública, um registro pouco presente nos outros sete documentos. Do lado
oposto, o vocabulário mais sub-representado no PBIA é de **implantação
técnico-industrial**: *smart* (0× vs. 106×, quase todo do AI+), *factory*
(0× vs. 69×, do AI Continent Action Plan), *accelerate*, *build*,
*breakthrough*, *deploy*, *manufacturing* — termos concentrados no registro
setorial-industrial da China e da UE que o PBIA não replica.

## 8.4. Preferência temática — nem "direitos humanos" nem "puro desenvolvimento" (Seção 6)

O perfil temático do próprio PBIA é **híbrido, sem um foco dominante**:
**30% Infraestrutura, 30% Desenvolvimento nacional, 19% Direitos/ética,
14% Governança, 6% Segurança**. Isso responde diretamente à pergunta sobre
foco: o PBIA **não** é dominado pela linguagem de direitos/ética como o
BRICS (36%) e sobretudo a OCDE (43% — quase metade do vocabulário temático
da recomendação é sobre direitos, equidade e responsabilidade); mas também
**não** é um documento puramente desenvolvimentista/de infraestrutura como
o China New Gen (39% desenvolvimento + 34% infraestrutura, só 8% direitos)
ou o AI Continent Action Plan (49% infraestrutura, só 7% direitos). O PBIA
ocupa uma posição intermediária nas cinco categorias ao mesmo tempo — não
por indiferença a nenhum eixo, mas por **balancear os cinco eixos de forma
mais uniforme** que qualquer um dos sete documentos de referência.

## 8.5. A nuance mais importante: vocabulário parecido ≠ perfil temático parecido

A similaridade de cosseno (8.1) diz que o PBIA soa mais como o **BRICS**.
Mas ao comparar a **forma** do perfil temático — não as palavras em si, e
sim como cada documento distribui sua ênfase pelas 5 categorias —, o
documento mais parecido com o PBIA é a **Apply AI Strategy (14,3 pontos
percentuais de distância)**, seguida de perto pelo **China New Gen (14,9
p.p.)**; o **BRICS fica em 6º lugar nessa métrica (26,2 p.p.)**, quase tão
distante quanto a própria OCDE (35,5 p.p.). Ou seja: o PBIA **usa** um
vocabulário mais parecido com o do BRICS (registro de desenvolvimento e
cooperação Sul-Sul), mas **organiza sua ênfase temática** de um jeito mais
parecido com o equilíbrio infraestrutura+desenvolvimento moderado da
estratégia europeia de aplicação e do plano-mestre chinês — sem reproduzir
o forte peso em direitos/governança do BRICS nem o desequilíbrio quase
total para infraestrutura da UE/China. É uma peça de evidência de que o
PBIA constrói uma posição própria, não a cópia de um único modelo.

## 8.6. Mapa de posicionamento consolidado (Seção 7)

No MDS (67% da variância explicada em 2D — portanto uma simplificação, não
uma reprodução exata das oito distâncias), o PBIA cai **próximo do centro
geométrico da constelação dos oito documentos**, e o ponto mais próximo por
distância euclidiana é o **America's AI Action Plan (0,044)** — mais perto,
nesse mapa consolidado, do que o BRICS (0,229) ou o próprio China New Gen
(0,232). Isso não contradiz a Seção 8.1 (onde o BRICS tem a maior
similaridade de cosseno *par a par* com o PBIA): o MDS reflete a
configuração de **todas** as distâncias simultaneamente, e a proximidade
com os EUA no mapa 2D é, em parte, um artefato de o America's AI Action
Plan não estar fortemente "puxado" para nenhum dos outros clusters (China
de um lado, UE do outro), sobrando-lhe uma posição mais central — geometria
de projeção, não um sinal de similaridade direta mais forte que a do BRICS.

## 8.7. Limitações que permanecem (declaradas, não escondidas)

- O **léxico temático** (Seção 6) é deduzido a priori pelo pesquisador, não
  extraído estatisticamente dos dados — outra escolha de palavras-raiz
  produziria números diferentes, embora a ordem de grandeza dos achados
  (PBIA como híbrido, OCDE/BRICS como documentos de direitos, China/UE como
  documentos de infraestrutura) seja consistente com o vocabulário
  distintivo da Seção 5, obtido por um método independente.
- O **stemmer** (Seção 2) é conservador fora do dicionário curado — verbos
  no gerúndio/passado não cobertos por ele permanecem não normalizados, o
  que pode subestimar levemente a similaridade real em alguns pares.
- O teste de **keyness** (Seção 5) compara o PBIA contra o *pool* dos sete
  documentos combinados, não par a par — identifica o que é distintivo do
  PBIA *no agregado*, não qual documento específico mais compartilha ou
  mais diverge de cada termo.